# Поиск неполных дублей FAQ ЕЛЧ
Положите рядом две актуальные выгрузки Excel и проверьте пути в первой ячейке. Запустите **Run All**. В `results/partial_run` появятся HTML с кандидатами и Excel для вычитки. Ноутбук работает локально. Ему нужны Python 3.10+, `numpy`, `scipy` и `openpyxl`. Для готовых эмбеддингов дополнительно нужен `pandas`.

Неполный дубль — карточки про один сценарий с эквивалентными ответами или с ответом, который можно расширить без потери отдельного условия. Выдача содержит кандидатов на вычитку. Текстовое сходство, номера, условия и классификация служат признаками. Полные совпадения открываются в одной вкладке.

In [ ]:
from pathlib import Path
from time import perf_counter

ROOT = Path.cwd()
EXPORT_FILES = [
    ROOT / "faq_main.xlsx",
    ROOT / "faq_additional.xlsx",
]
BENCHMARK_FILE = ROOT / "faq_partial_benchmark.csv"
OUTPUT_DIR = ROOT / "results" / "partial_run"

# Лёгкий режим по умолчанию. Укажите оба файла, чтобы добавить готовые векторы.
E5_NPY = None       # ROOT / "faq_embeddings_final.npy"
E5_SNAPSHOT = None  # ROOT / "faq_answers_snapshot.pkl" или исходный CSV
E5_MIN_COSINE = None   # После просмотра листа «Пороги E5».
E5_NEIGHBOUR_FLOOR = None  # Например, .80 для отдельного поиска по всему проверенному кэшу.
TOP_K = 12
ONLY_SAME_PRODUCT = False
LEXICAL_THRESHOLDS = dict(
    answer=.76, question=.46, coverage=.93, containment_question=.20,
    strong_answer=.95, same_question_answer=.44,
)

## Чтение и признаки
Сначала проверяем уникальность ID и берём активные FAQ ЕЛЧ. Косинус считается отдельно по TF‑IDF вопросов и ответов. Для каждой карточки есть собственный лимит соседей внутри и между продуктами. Точное совпадение с альтернативным вопросом добавляет пары. Повторы общего вопроса у более чем 24 FAQ пропускаются на этом шаге.

Направленное покрытие измеряет долю общих взвешенных токенов от длины каждого ответа. Текстовая вложенность дополнительно требует непрерывного совпадения последовательности токенов. Проверяем явные различия в программах, операциях, единицах, числах, запретах и ссылках. Косинусное расстояние равно `1 − cosine`.

In [ ]:
import re
from collections import Counter
from dataclasses import dataclass
from openpyxl import load_workbook

CONTEXT_FIELDS = ('Продукт', 'Подпродукт', 'Тематика', 'Основной канал',
                  'Дополнительный канал', 'Дата, время включения', 'Дата, время отключения')
MAIN, EXTRA = 'Основной канал', 'Дополнительный канал'
LEGACY = 'Перенесено из предыдущего отчёта'
SAME, DIFFERENT, UNKNOWN = 'same', 'different', 'unknown'
INSURANCE_PROGRAM = re.compile(r'\bзащит[аеуы]\s+(дома|денег)\b', re.I)
DEPOSIT_CLOSE = re.compile(r'^как\s+(?:досрочно\s+)?закрыть\s+вклад\s+[«"]([^»"]+)[»"]\s*\??$', re.I)
NUMBERS = re.compile(r'\d{1,3}(?:[ \u00a0]\d{3})+(?:[.,]\d+)?|\d+(?:[.,:/-]\d+)*')
NEGATION = re.compile(r'\b(?:не|нет|нельзя|без)\b', re.I)


def clean(value):
    return '' if value is None else str(value)


def norm(value):
    return ' '.join(clean(value).split())


def numbers(answer):
    return {re.sub(r'\s', '', token).replace(',', '.')
            for token in NUMBERS.findall(answer)}


def differences(a, b):
    links = {field for field in a.row.keys() | b.row.keys() if field and field.startswith('Ссылка')}
    fields = [field for field in (*CONTEXT_FIELDS, *sorted(links))
              if norm(a.row.get(field)) != norm(b.row.get(field))]
    if numbers(a.answer) != numbers(b.answer):
        fields.append('Числа в тексте')
    if Counter(NEGATION.findall(a.answer.casefold())) != Counter(NEGATION.findall(b.answer.casefold())):
        fields.append('Отрицания в тексте')
    return fields


@dataclass(frozen=True)
class FAQ:
    id: str
    question: str
    answer: str
    alternatives: tuple
    source: str
    row: dict

    @property
    def product_key(self):
        return tuple(norm(self.row.get(field)).casefold() for field in CONTEXT_FIELDS[:2])


def context(a, b):
    ap, asub = a.product_key
    bp, bsub = b.product_key
    if ap and bp and ap != bp:
        return DIFFERENT, 'Разные продукты'
    if not ap or not bp or not asub or not bsub:
        return UNKNOWN, 'У одной или обеих карточек не заполнен продукт или подпродукт'
    if asub != bsub:
        return DIFFERENT, 'Разные подпродукты'
    program = None
    if (ap, asub) == ('страхование', 'злс'):
        program = INSURANCE_PROGRAM
    elif ap == 'вклады и счета':
        program = DEPOSIT_CLOSE
    if program:
        left, right = (program.search(norm(card.question)) for card in (a, b))
        if left and right and norm(left[1]).casefold() != norm(right[1]).casefold():
            return DIFFERENT, 'Разные программы внутри подпродукта'
    return SAME, 'Совпадают продукт и подпродукт'


def read_records(paths):
    records, seen = {}, set()
    counts = Counter()
    required = {'Идентификатор', 'Вопрос', 'Ответ', 'Продукт', 'Подпродукт', 'Статус', MAIN, EXTRA}
    for path in paths:
        book = load_workbook(path, read_only=True, data_only=True)
        try:
            rows = iter(book['Выгрузка'].values)
            fields = next(rows)
            if missing := required.difference(fields):
                raise ValueError(f'{path.name}: отсутствуют поля {sorted(missing)}')
            alt_fields = [f for f in fields if f and f.startswith('Альтернативный вопрос')]
            for line, values in enumerate(rows, 2):
                if all(value is None for value in values):
                    continue
                counts['Строк в выгрузках'] += 1
                row = dict(zip(fields, values))
                ident = norm(row['Идентификатор'])
                if not ident or ident in seen:
                    raise ValueError(f'{path.name}:{line}: пустой или повторяющийся ID FAQ')
                seen.add(ident)
                main, extra = ('ЕЛЧ' in {norm(x) for x in clean(row[key]).split(',')}
                               for key in (MAIN, EXTRA))
                if norm(row['Статус']) != 'Активно' or not (main or extra):
                    continue
                if not norm(row['Вопрос']) or not norm(row['Ответ']):
                    raise ValueError(f'{path.name}:{line}: пустой вопрос или ответ')
                source = 'main' if main else 'additional'
                counts[source] += 1
                records[ident] = FAQ(ident, clean(row['Вопрос']), clean(row['Ответ']),
                                     tuple(clean(row[f]) for f in alt_fields if norm(row[f])),
                                     source, row)
        finally:
            book.close()
    return records, counts

In [ ]:
"""Lightweight, offline candidate search for incomplete FAQ duplicates."""

from collections import Counter, defaultdict
from dataclasses import dataclass
import math
from pathlib import Path
import re
import unicodedata

import numpy as np
from scipy.sparse import csr_matrix



TOKEN = re.compile(r'[^\W\d_]+|\d+(?:[.,:/-]\d+)*', re.UNICODE)
LIST_NUMBER = re.compile(r'(?m)^\s*\d{1,2}[.)]\s+')
RISK_NUMBER = re.compile(
    r'(?<!\w)(\d+(?:[ \u00a0]\d{3})*(?:[.,]\d+)?)\s*'
    r'(%|рабоч\w*\s+(?:дн\w*|день)|календар\w*\s+(?:дн\w*|день)|дн(?:я|ей)?\b|день\b|'
    r'руб\w*|₽|тыс\w*|млн\w*|мес\w*|год\w*|лет\b|час\w*)', re.I)
ARTICLE = re.compile(r'\b(статья|пункт|подпункт)\s*(\d+(?:\.\d+)*)', re.I)
QUOTES = re.compile(r'[«"]([^»"]+)[»"]')
NAMED_DEPOSIT = re.compile(r'\b(?:вклад|сч[её]т)\w*\s*[«"]([^»"]+)[»"]', re.I)
PRIME = re.compile(r'\bсберпрайм\b(?:\s*(?:\+|плюс))?', re.I)
SCOPE = re.compile(r'\b(?:сберперв\w*|сберпремьер\w*|доступн\w* выгод\w*|свой максимум|'
                   r'подушка безопасности|страхование по часам|дмс|защита от травм)\b')
URL = re.compile(r'https?://\S+', re.I)
QUOTE_CHARS = str.maketrans('', '', '«»„“”"')
DASH_CHARS = str.maketrans({'—': '–', '−': '–'})
OPERATIONS = [
    (r'\b(?:внос\w*|внести|полож(?!ен)\w*|пополн\w*)\b', r'\b(?:снят\w*|сним\w*|выдал|выдач\w*)\b', 'Внесение и выдача'),
    (r'\bподключ\w*\b', r'\bотключ\w*\b', 'Подключение и отключение'),
    (r'\bдобав\w*\b', r'\bудал\w*\b', 'Добавление и удаление'),
    (r'\bоткр\w*\b', r'\bзакр\w*\b', 'Открытие и закрытие'),
]
OPERATIONS = [(re.compile(a), re.compile(b), label) for a, b, label in OPERATIONS]
STRONG_NEGATION = re.compile(r'\b(?:нельзя|запрещ\w*|не\s+допуска\w*|не\s+предусмотр\w*)\b')


@dataclass(frozen=True)
class Thresholds:
    answer: float = .76
    question: float = .46
    coverage: float = .93
    containment_question: float = .20
    embedding: float | None = None
    strong_answer: float = .95
    same_question_answer: float = .44


def words(text):
    return TOKEN.findall(LIST_NUMBER.sub('', text).casefold().replace('ё', 'е'))


def answer_key(text):
    # Preserve units, +, digits, punctuation inside numbers, and case-sensitive URLs.
    if re.search(r'https?://', text, re.I):
        return norm(text)
    normalized = unicodedata.normalize('NFC', text).translate(QUOTE_CHARS).translate(DASH_CHARS)
    return norm(normalized.casefold().replace('ё', 'е')).removesuffix('.').rstrip()


def weighted_matrix(documents, max_df=.20):
    """Return unit TF-IDF rows for unigrams and two-word phrases."""
    count = len(documents)
    terms = [Counter([*tokens, *zip(tokens, tokens[1:])]) for tokens in documents]
    frequency = Counter(term for row in terms for term in row)
    vocab = {term: k for k, term in enumerate(
        term for term, df in frequency.items() if df <= max(2, count * max_df))}
    indptr, indices, data = [0], [], []
    for row in terms:
        weighted = [(vocab[t], (1 + math.log(n)) * (1 + math.log((1 + count) / (1 + frequency[t]))))
                    for t, n in row.items() if t in vocab]
        size = math.sqrt(sum(w * w for _, w in weighted))
        indices.extend(i for i, _ in weighted)
        data.extend(w / size for _, w in weighted)
        indptr.append(len(data))
    result = csr_matrix((np.asarray(data, dtype=np.float32), indices, indptr),
                        shape=(count, len(vocab)), dtype=np.float32)
    result.sort_indices()
    return result


class Corpus:
    def __init__(self, records):
        self.cards = [records[key] for key in sorted(records)]
        self.index = {card.id: i for i, card in enumerate(self.cards)}
        self.form = [answer_key(c.answer) for c in self.cards]
        answers = [words(c.answer) for c in self.cards]
        questions = [words(c.question) for c in self.cards]
        self.answer_matrix = weighted_matrix(answers)
        self.question_matrix = weighted_matrix(questions, max_df=.40)
        counts = Counter(w for tokens in answers for w in set(tokens))
        self.idf = {w: 1 + math.log((1 + len(answers)) / (1 + n)) for w, n in counts.items()}
        self.tokens = [Counter(tokens) for tokens in answers]
        self.token_text = [' ' + ' '.join(tokens) + ' ' for tokens in answers]
        self.token_count = list(map(len, answers))
        self.weight = [sum(self.idf[w] * n for w, n in ts.items()) for ts in self.tokens]
        self.question_forms = [{norm(q).casefold() for q in (c.question, *c.alternatives)}
                               for c in self.cards]
        self.product = np.array([c.product_key[0] for c in self.cards], dtype=object)

    def pool(self, top_k=12, block=96, answer_floor=.36, question_floor=.58, dense=None,
             dense_search_floor=None):
        """Keep independent neighbour budgets inside and across products for each signal."""
        if top_k < 1 or block < 1:
            raise ValueError('Размер блока и число соседей должны быть положительными.')
        seen = defaultdict(set)

        def select(row, columns, scores, tag):
            if not columns.size:
                return
            eligible = np.array([j != row and self.form[j] != self.form[row] for j in columns])
            same = (self.product[columns] == self.product[row]) & bool(self.product[row])
            for mask in (eligible & same, eligible & ~same):
                offsets = np.nonzero(mask)[0]
                if offsets.size > top_k:
                    order = np.lexsort((columns[offsets], -scores[offsets]))[:top_k]
                    offsets = offsets[order]
                for other in columns[offsets]:
                    seen[tuple(sorted((row, int(other))))].add(tag)

        for matrix, floor, tag in ((self.answer_matrix, answer_floor, 'ответы'),
                                   (self.question_matrix, question_floor, 'вопросы')):
            transpose = matrix.T
            for start in range(0, len(self.cards), block):
                batch = (matrix[start:start + block] @ transpose).tocsr()
                for offset in range(batch.shape[0]):
                    lo, hi = batch.indptr[offset:offset + 2]
                    columns, scores = batch.indices[lo:hi], batch.data[lo:hi]
                    good = scores >= floor
                    select(start + offset, columns[good], scores[good], tag)

        # Named question overlaps, including alternatives, rescue some paraphrases with low lexical cosine.
        inverted = defaultdict(list)
        for row, forms in enumerate(self.question_forms):
            for form in forms:
                if len(words(form)) >= 3:
                    inverted[form].append(row)
        for rows in inverted.values():
            if len(rows) > 24:
                continue
            for n, i in enumerate(rows):
                for j in rows[n + 1:]:
                    if self.form[i] != self.form[j]:
                        seen[i, j].add('та же формулировка вопроса')

        if dense_search_floor is not None:
            if dense is None:
                raise ValueError('Для поиска соседей E5 нужен проверенный кэш.')
            vectors, positions = dense
            keys = np.array(sorted(positions))
            if keys.size:
                sample = vectors[[positions[int(i)] for i in keys]]
                for start in range(0, len(keys), block):
                    scores = sample[start:start + block] @ sample.T
                    for offset, score in enumerate(scores):
                        keep = score >= dense_search_floor
                        select(int(keys[start + offset]), keys[keep], score[keep], 'E5')
        return seen

    def features(self, pairs, dense=None):
        keys = sorted(pairs)
        for start in range(0, len(keys), 1024):
            batch = keys[start:start + 1024]
            ai, bi = np.asarray(batch).T
            score_a = np.asarray(self.answer_matrix[ai].multiply(self.answer_matrix[bi]).sum(axis=1)).ravel()
            score_q = np.asarray(self.question_matrix[ai].multiply(self.question_matrix[bi]).sum(axis=1)).ravel()
            for (i, j), a_score, q_score in zip(batch, score_a, score_q):
                a, b = self.cards[i], self.cards[j]
                shared = sum(self.idf[w] * n for w, n in (self.tokens[i] & self.tokens[j]).items())
                cov_a = shared / self.weight[i] if self.weight[i] else 0
                cov_b = shared / self.weight[j] if self.weight[j] else 0
                same_question = bool(self.question_forms[i] & self.question_forms[j])
                dense_score = None
                if dense is not None:
                    v, pos = dense
                    if i in pos and j in pos:
                        dense_score = float(np.clip(v[pos[i]] @ v[pos[j]], -1, 1))
                score = {
                    'a': a.id, 'b': b.id, 'answer': min(1., float(a_score)),
                    'question': min(1., float(q_score)), 'same_question': same_question,
                    'cover_a': min(1., cov_a), 'cover_b': min(1., cov_b), 'embedding': dense_score,
                    'literal': min(self.token_count[i], self.token_count[j]) >= 8
                    and (self.token_text[i] in self.token_text[j] or self.token_text[j] in self.token_text[i]),
                }
                yield (i, j), score


def _numbers_with_units(text):
    result = defaultdict(set)
    for value, unit in RISK_NUMBER.findall(text.casefold()):
        unit = norm(unit)
        if unit.startswith(('рабоч', 'календар')):
            unit = unit.split()[0][:5] + ' дни'
        elif unit.startswith(('дн', 'ден')):
            unit = 'дни'
        elif unit == 'лет':
            unit = 'год'
        else:
            unit = unit[:3]
        result[unit].add(float(value.replace(' ', '').replace('\u00a0', '').replace(',', '.')))
    return result


def risks(a, b):
    relation, subtype = context(a, b)
    qa, qb = [norm(x.question).casefold().replace('ё', 'е') for x in (a, b)]
    qa_refs = {(k.casefold(), n) for k, n in ARTICLE.findall(qa)}
    qb_refs = {(k.casefold(), n) for k, n in ARTICLE.findall(qb)}
    hard, caution = [], []
    same_product = bool(a.product_key[0] and a.product_key[0] == b.product_key[0])
    if relation == 'different' and 'внутри' in subtype:
        hard.append(subtype)
    elif relation != 'same':
        caution.append(subtype)
    if any({n for k2, n in qa_refs if k2 == k}.isdisjoint({n for k2, n in qb_refs if k2 == k})
           for k in set(k for k, _ in qa_refs) & set(k for k, _ in qb_refs)):
        hard.append('Разные статьи или пункты')
    names_a = {norm(s) for s in QUOTES.findall(qa)}
    names_b = {norm(s) for s in QUOTES.findall(qb)}
    if same_product and a.product_key[0] == 'вклады и счета':
        names_a = names_a or {norm(s) for s in NAMED_DEPOSIT.findall(a.answer[:160].casefold())}
        names_b = names_b or {norm(s) for s in NAMED_DEPOSIT.findall(b.answer[:160].casefold())}
    prime_a, prime_b = PRIME.search(qa), PRIME.search(qb)
    if prime_a and prime_b and (prime_a[0] == 'сберпрайм') != (prime_b[0] == 'сберпрайм'):
        hard.append('Разные пакеты СберПрайм')
    scope_a, scope_b = set(SCOPE.findall(qa)), set(SCOPE.findall(qb))
    if scope_a and scope_b and scope_a.isdisjoint(scope_b):
        caution.append('Отличается явно названная программа или пакет')
    named_scope = a.product_key[0] in ('вклады и счета', 'страхование', 'инвестиции и ценные бумаги ук')
    if same_product and named_scope and names_a and names_b and names_a.isdisjoint(names_b):
        hard.append('Разные явные названия продукта или программы')
    for x, y, reason in OPERATIONS:
        a_action, b_action = (bool(x.search(qa)), bool(y.search(qa))), (bool(x.search(qb)), bool(y.search(qb)))
        if (a_action, b_action) in (((True, False), (False, True)), ((False, True), (True, False))):
            hard.append('Разные операции: ' + reason)
    na, nb = _numbers_with_units(a.answer), _numbers_with_units(b.answer)
    for unit in sorted(na.keys() & nb.keys()):
        if na[unit].isdisjoint(nb[unit]):
            hard.append(f'Разные значения ({unit}): {sorted(na[unit])} ↔ {sorted(nb[unit])}')
        elif na[unit] != nb[unit]:
            caution.append(f'Дополнительные числовые условия ({unit})')
    if bool(STRONG_NEGATION.search(a.answer.casefold())) != bool(STRONG_NEGATION.search(b.answer.casefold())):
        caution.append('Явный запрет только в одном ответе')
    if (set(URL.findall(a.answer)) != set(URL.findall(b.answer))
            or any(flag.startswith('Ссылка') for flag in differences(a, b))):
        caution.append('Разные ссылки')
    return same_product, relation, hard, caution


def qualify(feature, flags, cut, same_product_only=False):
    same, relation, hard, caution = flags
    if same_product_only and not same:
        return 'вне продукта', ''
    question = feature['question']
    answer = feature['answer']
    exact_q = feature['same_question']
    containment = max(feature['cover_a'], feature['cover_b'])
    lexical = answer >= cut.answer and (question >= cut.question or exact_q)
    same_form = exact_q and answer >= cut.same_question_answer
    strong = answer >= cut.strong_answer
    literal = feature.get('literal', False) and answer >= .45
    covers = containment >= cut.coverage and answer >= .23 and (question >= cut.containment_question or exact_q)
    embedding = (cut.embedding is not None and feature['embedding'] is not None
                 and feature['embedding'] >= cut.embedding
                 and (question >= cut.question or exact_q or answer >= cut.answer))
    if not (lexical or same_form or strong or covers or embedding or literal):
        return 'ниже порога', ''
    signals = [(lexical, 'TF-IDF вопроса и ответа'), (same_form, 'общий вопрос'),
               (strong, 'высокий cosine ответа'), (covers, 'покрытие'),
               (literal, 'текстовая вложенность'), (embedding, 'E5')]
    method = ', '.join(name for active, name in signals if active)
    if hard:
        return 'существенное различие', method
    strong_question = ((exact_q and answer >= .65) or (question >= .65 and answer >= .78)
                       or (question >= .58 and containment >= .96))
    if caution or relation != 'same' or not strong_question:
        return 'сверка контекста', method
    return 'приоритетная вычитка', method


def load_embedding_cache(path, snapshot, corpus):
    """Match row positions from the original run to current cards by ID and answer text."""
    import pandas as pd

    if Path(snapshot).suffix.lower() == '.csv':
        frame = pd.read_csv(snapshot, dtype=str, keep_default_na=False, encoding='utf-8-sig')
    elif Path(snapshot).suffix.lower() == '.pkl':
        frame = pd.read_pickle(snapshot)
    else:
        raise ValueError('Доступны CSV со столбцами Идентификатор и Ответ или доверенный локальный PKL.')
    ids = frame['Идентификатор'].fillna('').astype(str).str.strip()
    answers = frame['Ответ'].fillna('').astype(str).str.strip()
    if ids.empty or ids.eq('').any() or ids.duplicated().any() or answers.eq('').any():
        raise ValueError('В снимке кэша пустой ответ или пустой/повторный ID.')
    vectors = np.load(path, mmap_mode='r', allow_pickle=False)
    if vectors.ndim != 2 or vectors.shape[0] != len(ids) or vectors.shape[1] < 1 or vectors.dtype.kind != 'f':
        raise ValueError('Размерность или тип NPY не соответствует снимку.')
    matches = [(i, corpus.index[k]) for i, (k, a) in enumerate(zip(ids, answers))
               if k in corpus.index and corpus.cards[corpus.index[k]].answer.strip() == a]
    if not matches:
        raise ValueError('Ни одного совпадения ID и ответа между выгрузками и снимком.')
    a, b = np.asarray(matches).T
    selected = vectors[a].astype(np.float32)
    lengths = np.linalg.norm(selected, axis=1)
    if not np.isfinite(lengths).all() or (lengths == 0).any():
        raise ValueError('Кэш содержит нулевые или нечисловые векторы среди совпавших FAQ.')
    selected /= lengths[:, None]
    return selected, dict(zip(b.tolist(), range(len(b))))

In [ ]:
"""Benchmark validation, queue construction, and offline reports for FAQ review."""

from collections import Counter, defaultdict
import csv
from hashlib import sha256
from html import escape
import json
from pathlib import Path
from tempfile import TemporaryDirectory

from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter



POSITIVE = {'дубль', 'покрытие'}
REVIEW = {'приоритетная вычитка', 'сверка контекста'}
SEVERITY = {'приоритетная вычитка': 0, 'сверка контекста': 1, 'существенное различие': 2}


def fingerprint(card):
    payload = [card.question, *card.alternatives, card.answer, *card.product_key]
    return sha256(json.dumps(payload, ensure_ascii=False).encode()).hexdigest()


def load_benchmark(path, corpus):
    if path is None:
        return {}, 0
    with Path(path).open(encoding='utf-8-sig', newline='') as stream:
        rows = list(csv.DictReader(stream))
    good, stale = {}, 0
    for row in rows:
        a, b = row['id_a'], row['id_b']
        if (a not in corpus.index or b not in corpus.index
                or fingerprint(corpus.cards[corpus.index[a]]) != row['hash_a']
                or fingerprint(corpus.cards[corpus.index[b]]) != row['hash_b']):
            stale += 1
            continue
        key = tuple(sorted((corpus.index[a], corpus.index[b])))
        if key in good:
            raise ValueError(f'Повторная разметка пары {a} и {b}.')
        good[key] = row
    return good, stale


def search(corpus, pool, benchmark, thresholds, dense=None):
    combined = dict(pool)
    combined.update({key: set() for key in benchmark if key not in pool})
    found, benchmark_features = [], {}
    for key, feature in corpus.features(combined, dense):
        a, b = (corpus.cards[i] for i in key)
        if key not in benchmark and qualify(feature, (True, 'same', [], []), thresholds)[0] == 'ниже порога':
            continue
        flags = risks(a, b)
        row = dict(feature, cards=(a, b), flags=flags, sources=sorted(combined[key]), key=key)
        row['queue'], row['signals'] = qualify(feature, flags, thresholds)
        row['audit'] = benchmark.get(key, {})
        if key in benchmark:
            benchmark_features[key] = row
        if row['sources'] and row['queue'] in SEVERITY and answer_key(a.answer) != answer_key(b.answer):
            if row['audit'].get('decision') in ('разные', 'конфликт'):
                row = {**row, 'queue': 'существенное различие'}
            found.append(row)

    def order(row):
        a, b = row['cards']
        label = row['audit'].get('decision', '')
        audit_rank = 0 if label in POSITIVE else 2 if label in ('разные', 'конфликт') else 1
        score = .60 * row['answer'] + .22 * row['question'] + .18 * max(row['cover_a'], row['cover_b'])
        return SEVERITY[row['queue']], audit_rank, -score, a.id, b.id

    return sorted(found, key=order), benchmark_features


def benchmark_rows(features, cut, e5_only=False, splits=('подбор', 'проверка', 'контрольный аудит')):
    result = []
    for split in splits:
        rows = [r for r in features.values()
                if r['audit']['split'] == split and r['audit']['decision'] != 'уточнить'
                and answer_key(r['cards'][0].answer) != answer_key(r['cards'][1].answer)
                and (not e5_only or r['embedding'] is not None)]
        if not rows:
            continue
        for method in ('Только cosine ответа', 'Признаки · внутри продукта', 'Признаки · все продукты'):
            tp, fp, fn = 0, 0, 0
            for row in rows:
                if method == 'Только cosine ответа':
                    selected = bool(row['sources']) and row['answer'] >= cut.answer
                else:
                    status, _ = qualify(row, row['flags'], cut, method == 'Признаки · внутри продукта')
                    selected = bool(row['sources']) and status in REVIEW
                true = row['audit']['decision'] in POSITIVE
                tp += int(selected and true)
                fp += int(selected and not true)
                fn += int(not selected and true)
            result.append((split, method, len(rows), tp, fp, fn,
                           tp / (tp + fp) if tp + fp else None,
                           tp / (tp + fn) if tp + fn else None))
    return result


def lexical_sweep(features, current):
    from dataclasses import replace
    from itertools import product

    grid = product((.70, .76, .82), (.34, .46, .56), (.90, .93, .96))
    rows = []
    for answer, question, coverage in grid:
        cut = replace(current, answer=answer, question=question, coverage=coverage, embedding=None)
        stats = benchmark_rows(features, cut, splits=('подбор',))
        for item in stats:
            if item[1] == 'Признаки · все продукты':
                rows.append((answer, question, coverage, *item[2:]))
    return rows


def embedding_sweep(features, lexical_cut):
    """Compare E5 thresholds on the same labelled pairs with both vectors present."""
    from dataclasses import replace

    grid = (None, .80, .83, .86, .89, .92, .95, .97)
    result = []
    for cutoff in grid:
        run = benchmark_rows(features, replace(lexical_cut, embedding=cutoff), e5_only=True)
        result.extend((cutoff, *r) for r in run if r[1] == 'Признаки · все продукты')
    return result


def exact_report(corpus):
    grouped = defaultdict(list)
    for k, card in zip(corpus.form, corpus.cards):
        grouped[k].append(card)
    rows = []
    for cards in grouped.values():
        if len(cards) < 2:
            continue
        for card in cards:
            rows.append((len(rows) + 1, cards[0].id, len(cards), card.id, card.row['Продукт'],
                         card.row['Подпродукт'] or '', card.question, card.answer))
    return rows


def view_row(r):
    a, b = r['cards']
    same, relation, hard, soft = r['flags']
    audit = r['audit']
    queue = r['queue']
    if not r['sources']:
        queue = 'вне пула соседей'
    if answer_key(a.answer) == answer_key(b.answer):
        queue = 'совпадает после нормализации'
    return (queue, 'один продукт' if same else 'между продуктами или продукт пуст',
            r['answer'], r['question'], float(r['cover_a']), float(r['cover_b']), r['embedding'],
            r['signals'], ' | '.join([*hard, *soft]), audit.get('decision', ''),
            audit.get('comment', ''), audit.get('old_verdict', ''), a.id, b.id,
            a.row['Продукт'], a.row['Подпродукт'] or '', b.row['Продукт'], b.row['Подпродукт'] or '',
            a.question, b.question, a.answer, b.answer, ' | '.join(r['sources']), audit.get('source', ''))


PAIR_HEADER = ('Очередь', 'Сравнение', 'Cosine ответа (TF-IDF)', 'Cosine вопроса (TF-IDF)',
               'Доля A в B', 'Доля B в A', 'Cosine E5', 'Сигналы', 'Что проверить',
               'Решение по вычитанной паре', 'Комментарий', 'Старый вердикт ГигаЧата',
               'ID A', 'ID B', 'Продукт A', 'Подпродукт A', 'Продукт B', 'Подпродукт B',
               'Вопрос A', 'Вопрос B', 'Ответ A', 'Ответ B', 'Источник поиска', 'Источник разметки')
EXACT_HEADER = ('Строка', 'Группа', 'Размер', 'ID FAQ', 'Продукт', 'Подпродукт', 'Вопрос', 'Ответ')
METRIC_HEADER = ('Раздел', 'Метод', 'Размеченных пар', 'Найдено полезных', 'Лишних', 'Пропущено полезных',
                 'Доля полезных среди найденных', 'Доля найденных среди полезных в выборке')


def write_workbook(path, summaries, candidates, differences, audited, exact, metrics, lexical, sweep):
    wb = Workbook()
    wb.remove(wb.active)

    def sheet(name, header, data, widths, score_cols=(), row_height=34):
        ws = wb.create_sheet(name)
        ws.append(header)
        for row in data:
            ws.append(row)
            for cell in ws[ws.max_row]:
                if isinstance(cell.value, str):
                    cell.data_type = 's'
            ws.row_dimensions[ws.max_row].height = row_height
        ws.freeze_panes = 'A2'
        ws.auto_filter.ref = ws.dimensions
        ws.sheet_view.showGridLines = False
        ws.row_dimensions[1].height = 34
        for cell in ws[1]:
            cell.fill = PatternFill('solid', fgColor='233F53')
            cell.font = Font(bold=True, color='FFFFFF')
            cell.alignment = Alignment(vertical='center', wrap_text=True)
        for n, width in enumerate(widths, 1):
            ws.column_dimensions[get_column_letter(n)].width = width
        for row in ws.iter_rows(min_row=2):
            for cell in row:
                if cell.column in score_cols:
                    cell.number_format = '0.000'
                cell.alignment = Alignment(vertical='top', wrap_text=True)
        return ws

    sheet('Сводка', ('Показатель', 'Значение'), summaries, (43, 98))
    sizes = (23, 23, 18, 18, 13, 13, 13, 32, 52, 20, 50, 53,
             34, 34, 24, 24, 24, 24, 47, 47, 65, 65, 26, 24)
    sheet('Неполные дубли', PAIR_HEADER, map(view_row, candidates), sizes, range(3, 8), 70)
    sheet('Существенные различия', PAIR_HEADER, map(view_row, differences), sizes, range(3, 8), 58)
    sheet('Аудит и разметка', PAIR_HEADER, map(view_row, audited), sizes, range(3, 8), 68)
    sheet('Полные совпадения', EXACT_HEADER, exact, (12, 35, 12, 35, 24, 25, 53, 86), (), 46)
    sheet('Проверка порогов', METRIC_HEADER, metrics, (18, 39, 16, 18, 14, 21, 24, 24), (7, 8))
    if lexical:
        sheet('Сетка TF-IDF', ('Порог ответа', 'Порог вопроса', 'Порог покрытия', *METRIC_HEADER[2:]),
              lexical, (16, 16, 17, 16, 18, 14, 21, 24, 24), (1, 2, 3, 8, 9))
    if sweep:
        sheet('Пороги E5', ('Порог E5', *METRIC_HEADER), sweep,
              (14, 18, 39, 16, 18, 14, 21, 24, 24), (1, 8, 9))
    wb.save(path)


def write_html(path, summaries, candidates, differences, exact, metrics):
    def esc(x):
        return escape(str(x), quote=True)

    def pair_table(rows):
        out = ['<table><thead><tr><th>Пара и ответы</th><th>Очередь</th>'
               '<th>Cosine ответа / вопроса</th><th>Покрытие A / B</th><th>Замечания</th></tr></thead><tbody>']
        for r in rows:
            a, b = r['cards']
            same, _, hard, soft = r['flags']
            audit = r['audit']
            ids = esc(a.id + ' · ' + b.id)
            product = esc(' / '.join((a.row['Продукт'] or '—', b.row['Продукт'] or '—')))
            notes = ' · '.join(filter(None, [*hard, *soft, audit.get('comment', '')]))
            e5 = '' if r['embedding'] is None else f'<br>E5 {r["embedding"]:.3f}'
            info = esc(audit.get('decision', ''))
            out.append(
                f'<tr data-scope="{int(same)}" data-queue="{esc(r["queue"])}"><td><small>{product}</small>'
                f'<details><summary>{esc(a.question)}<br>{esc(b.question)}</summary>'
                f'<p><small>{ids}</small></p><p><b>Ответ A</b><br>{esc(a.answer)}</p>'
                f'<p><b>Ответ B</b><br>{esc(b.answer)}</p></details></td>'
                f'<td>{esc(r["queue"])}<br><small>{info}</small></td>'
                f'<td>{r["answer"]:.3f} / {r["question"]:.3f}{e5}</td>'
                f'<td>{r["cover_a"]:.3f} / {r["cover_b"]:.3f}</td><td>{esc(notes)}</td></tr>')
        out.append('</tbody></table>')
        return ''.join(out)

    def simple_table(header, rows, attrs=False):
        return ('<table><thead><tr>' + ''.join(f'<th>{esc(x)}</th>' for x in header) +
                '</tr></thead><tbody>' + ''.join(
                    '<tr' + (' data-scope="" data-queue=""' if attrs else '') + '>' + ''.join(
                        f'<td>{esc(x if x is not None else "—")}</td>' for x in row) + '</tr>' for row in rows) +
                '</tbody></table>')

    body = ['<!doctype html><html lang="ru"><meta charset="utf-8"><meta name="viewport" '
            'content="width=device-width, initial-scale=1"><title>Неполные дубли FAQ</title>',
            """<style>
:root{color-scheme:light;--ink:#19334a;--muted:#516175;--line:#d9e2ea;--bg:#f4f7fa;--accent:#067385}
*{box-sizing:border-box}body{max-width:1440px;margin:auto;padding:24px;background:var(--bg);color:var(--ink);
font:15px/1.5 system-ui,sans-serif}h1{font-size:clamp(28px,5vw,44px);margin:0}h2{margin:.6em 0}
nav,.tools{display:flex;flex-wrap:wrap;gap:8px;margin:16px 0}
button,input,select{font:inherit;border:1px solid #aabac8;border-radius:7px;padding:8px 12px;background:white;color:var(--ink)}
button{cursor:pointer}button[aria-pressed=true]{background:var(--accent);color:white;border-color:var(--accent)}
input{flex:1;min-width:18ch}small{color:var(--muted)}
table{width:100%;border-collapse:collapse;background:white}th,td{text-align:left;padding:9px 12px;
vertical-align:top;border-bottom:1px solid var(--line)}th{background:#e4edf4}td:first-child{min-width:16ch}
summary{cursor:pointer;font-weight:650}details p{white-space:pre-wrap;max-height:28em;overflow:auto}section{overflow:auto}
[hidden]{display:none!important}
</style>""",
            '<h1>Неполные дубли FAQ ЕЛЧ</h1><p>Совпадения и покрытие ответов для редакторской проверки.</p>',
            '<nav><button data-tab="candidates" aria-pressed="true">Кандидаты</button>'
            '<button data-tab="differences" aria-pressed="false">Существенные различия</button>'
            '<button data-tab="exact" aria-pressed="false">Полные совпадения</button>'
            '<button data-tab="quality" aria-pressed="false">Проверка порогов</button></nav>',
            '<div class="tools"><input id="search" aria-label="Поиск" placeholder="Поиск по вопросу, ответу, продукту или ID">'
            '<select id="scope" aria-label="Продукт"><option value="">Все продукты</option>'
            '<option value="1">Внутри продукта</option><option value="0">Между продуктами или продукт пуст</option></select>'
            '<select id="queue" aria-label="Очередь"><option value="">Вся очередь</option>'
            '<option>приоритетная вычитка</option><option>сверка контекста</option></select></div>',
            '<p id="shown" aria-live="polite"></p>',
            '<section id="candidates">' + pair_table(candidates) + '</section>',
            '<section id="differences" hidden>' + pair_table(differences) + '</section>',
            '<section id="exact" hidden>' + simple_table(EXACT_HEADER, exact, True) + '</section>',
            '<section id="quality" hidden><h2>Сводка запуска</h2>' + simple_table(('Показатель','Значение'), summaries)
            + '<h2>Вычитанная выборка</h2>' + simple_table(METRIC_HEADER,
                [(*r[:6], *(None if x is None else f'{x:.1%}' for x in r[6:])) for r in metrics]) + '</section>',
            """<script>
const byId = id => document.getElementById(id);
let tab = 'candidates';
function refresh() {
  byId('shown').hidden = tab === 'quality';
  if (tab === 'quality') return;
  const q = byId('search').value.toLocaleLowerCase('ru').trim();
  const scope = byId('scope').value, queue = byId('queue').value;
  let n = 0;
  for (const row of byId(tab).querySelectorAll('tbody tr')) {
    row.hidden = (q && !row.textContent.toLocaleLowerCase('ru').includes(q))
      || (tab !== 'exact' && scope && row.dataset.scope !== scope)
      || (tab === 'candidates' && queue && row.dataset.queue !== queue);
    if (!row.hidden) n++;
  }
  byId('shown').textContent = 'Показано строк: ' + n;
}
for (const button of document.querySelectorAll('[data-tab]')) {
  button.addEventListener('click', () => {
    byId(tab).hidden = true;
    tab = button.dataset.tab;
    byId(tab).hidden = false;
    for (const item of document.querySelectorAll('[data-tab]')) {
      item.setAttribute('aria-pressed', String(item === button));
    }
    byId('search').disabled = tab === 'quality';
    byId('scope').disabled = tab === 'quality' || tab === 'exact';
    byId('queue').disabled = tab !== 'candidates';
    refresh();
  });
}
for (const id of ['search', 'scope', 'queue']) byId(id).addEventListener('input', refresh);
refresh();
</script></html>"""]
    Path(path).write_text(''.join(body), encoding='utf-8')


def export_review(output_dir, corpus, found, benchmark_features, metrics, lexical, sweep, thresholds, pool_size, stale):
    candidates = [r for r in found if r['queue'] in REVIEW]
    differences = [r for r in found if r['queue'] == 'существенное различие']
    counts = Counter(r['queue'] for r in candidates)
    exact = exact_report(corpus)
    audited = list(benchmark_features.values())
    audited.sort(key=lambda r: (r['audit']['source'] != 'прежний HTML', int(r['audit']['number'])))
    summaries = [
        ('Активных FAQ ЕЛЧ', len(corpus.cards)),
        ('Пар в пуле соседей', pool_size),
        ('Приоритетная вычитка', counts['приоритетная вычитка']),
        ('Сверка контекста', counts['сверка контекста']),
        ('Кандидатов между продуктами / с пустым продуктом', sum(not r['flags'][0] for r in candidates)),
        ('Существенное различие', len(differences)),
        ('Сохранённых вердиктов ГигаЧата', sum(bool(r['audit'].get('old_verdict')) for r in audited)),
        ('Групп полных совпадений', len({row[1] for row in exact})),
        ('Устаревших пар разметки', stale),
        ('Cosine ответа / вопроса', f'{thresholds.answer:.2f} / {thresholds.question:.2f}'),
        ('Покрытие / порог по вопросу', f'{thresholds.coverage:.2f} / {thresholds.containment_question:.2f}'),
        ('Высокий cosine ответа', thresholds.strong_answer),
        ('Cosine ответа при общей формулировке вопроса', thresholds.same_question_answer),
        ('Порог cosine E5', thresholds.embedding if thresholds.embedding is not None else 'не используется'),
        ('Покрытие A в B', 'Доля совпадающих взвешенных слов A относительно всех слов A. Это признак, направление проверяется по тексту.'),
        ('Метрики', 'Вычитанные выборки для аудита, без общих ID между разделами. Контрольный аудит отобран по очередям. Метрики по всей базе неизвестны.'),
        ('Полные совпадения', 'Нормализация пробелов, регистра, ё/е и оформления кавычек. Условие и вопрос проверяются отдельно.'),
        ('Действие', 'Кандидаты для проверки. Автоматических объединений и изменений в FAQ нет.'),
    ]
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    with TemporaryDirectory(dir=output_dir) as temp:
        temp = Path(temp)
        report, book = temp/'faq_partial_report.html', temp/'faq_partial_review.xlsx'
        write_html(report, summaries, candidates, differences, exact, metrics)
        write_workbook(book, summaries, candidates, differences, audited, exact, metrics, lexical, sweep)
        report.replace(output_dir/report.name)
        book.replace(output_dir/book.name)
    return summaries

## Запуск и проверка
Разметка от предыдущего аудита подхватывается, если указанный CSV доступен и совпадают отпечатки карточек. «Покрытие» и «дубль» считаются полезными кандидатами. Неоднозначные случаи исключены из сравнения порогов. Тесты по вычитанным парам помогают оценить очереди и ошибки. Общая полнота поиска по FAQ требует независимого эталона.

Векторный режим использует только готовый массив и снимок из того же запуска. Проверяйте пары по ID и неизменившемуся ответу. Включение `E5_NEIGHBOUR_FLOOR` добавляет проход по всему готовому кэшу батчами. Для этого прохода новое получение эмбеддингов и GPU не нужны. Сетка TF‑IDF рассчитывается на части для подбора. Сравнивайте пороги E5 на листе «Пороги E5», если есть совпадения с разметкой.

In [ ]:
started = perf_counter()
missing = [p for p in EXPORT_FILES if not p.is_file()]
if missing:
    raise FileNotFoundError("Уточните пути EXPORT_FILES: " + ", ".join(map(str, missing)))
if bool(E5_NPY) != bool(E5_SNAPSHOT) or ((E5_MIN_COSINE is not None or E5_NEIGHBOUR_FLOOR is not None) and not E5_NPY):
    raise ValueError("Укажите оба файла готового кэша E5.")
for value in (E5_MIN_COSINE, E5_NEIGHBOUR_FLOOR):
    if value is not None and not 0 <= value <= 1:
        raise ValueError("Пороги должны быть от 0 до 1.")

records, counts = read_records(EXPORT_FILES)
corpus = Corpus(records)
benchmark, stale = load_benchmark(BENCHMARK_FILE if BENCHMARK_FILE.is_file() else None, corpus)
dense = load_embedding_cache(E5_NPY, E5_SNAPSHOT, corpus) if E5_NPY else None
if dense is not None:
    print(f"Проверенные векторы E5: {len(dense[1])}/{len(corpus.cards)}")
thresholds = Thresholds(embedding=E5_MIN_COSINE, **LEXICAL_THRESHOLDS)
pool = corpus.pool(top_k=TOP_K, dense=dense, dense_search_floor=E5_NEIGHBOUR_FLOOR)
result, audited = search(corpus, pool, benchmark, thresholds, dense)
metrics = benchmark_rows(audited, thresholds)
lexical = lexical_sweep(audited, thresholds)
if ONLY_SAME_PRODUCT:
    result = [row for row in result if row["flags"][0]]
sweep = embedding_sweep(audited, thresholds) if dense is not None else []
print(f"Поиск и оценка заняли {perf_counter() - started:.1f} с.")
summary = export_review(OUTPUT_DIR, corpus, result, audited, metrics, lexical, sweep, thresholds, len(pool), stale)
for key, value in summary[:9]:
    print(f"{key}: {value}")
print(f"Отчёты: {OUTPUT_DIR.resolve()}")
print(f"Время с выгрузкой: {perf_counter() - started:.1f} с.")